Dec-POMDP Formulation & CTDE Architecture for MAPPO

In single-agent RL (PPO, SAC), the environment is assumed to be stationary: given state $s$ and action $a$, the probability distribution over the next state $s'$ is fixed.

Think of it like a video game

There are 2 agents:

        🌍 ENVIRONMENT

      A        B       TARGET
      🤖       🤖         🎯

The environment knows everything:

A's position
B's position
target position
walls
etc.

Call that state:

$$ s_t $$
What does Agent A receive?

Only the information we choose to show Agent A.

That's its observation:

$$ \boxed{o_A} $$

For example:

Environment knows:
A = (2,2)
B = (7,5)
Target = (9,9)

Agent A receives:
A's position = (2,2)
Target = (9,9)

So:

$$ o_A=[(2,2),(9,9)] $$

Maybe B isn't included.

What does Agent B receive?

B gets its own view:

Agent B receives:
B's position = (7,5)
Target = (9,9)

So:

$$ o_B=[(7,5),(9,9)] $$
Therefore
                 ENVIRONMENT
              knows everything
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
       Agent A               Agent B
       gets o_A               gets o_B
          ↓                     ↓
       action A               action B

\(o_A\) literally means "what Agent A can see."

\(o_B\) literally means "what Agent B can see."

Nothing more complicated.

Why not just call them \(s_A\) and \(s_B\)?

Because they're not the environment's state.

There is one actual world:

$$ s_t $$

But different agents can receive different pieces/views of it:

$$ o_A = O_A(s_t) $$ $$ o_B = O_B(s_t) $$

For example:

$$ s_t = [\underbrace{A}_{2,2}, \underbrace{B}_{7,5}, \underbrace{Target}_{9,9}] $$

but:

$$ o_A=[A,Target] $$ $$ o_B=[B,Target] $$

That's literally the whole deal.

And if we give A information about B?

Then simply:

$$ o_A=[A,B,Target] $$

Now A can see B.

That's completely allowed. The observation definition is part of our environment design.

# 1. Start with the actual problem

Imagine two robots need to push a box through a door.

```text
        DOOR
         ↓
     ┌────────┐
     │  BOX   │
     └────────┘

   🤖 A       🤖 B
```

Both robots need to cooperate.

Now suppose A can only see:

```text
A's camera:
    BOX
     ↑
    🤖A
```

And B can only see its own surroundings.

Your question is:

> **If A doesn't know what B is doing, HOW THE HELL can A coordinate with B?**

Exactly.

## Answer: it doesn't necessarily need to know B's full state.

There are several ways coordination can happen.

---

# 2. Simplest case: the environment itself gives enough information

Suppose the rules are:

> **If A pushes left, B should push right.**

A doesn't need to know B's exact position.

It can learn:

```text
A sees:
box is on my right
        ↓
A → push right
```

B independently sees:

```text
box is on my left
        ↓
B → push left
```

They coordinate because their **local observations + learned policies + shared objective** produce compatible actions.

Think of two people carrying a table.

Person A doesn't necessarily need a live dashboard saying:

```text
B's exact hand position = (1.72, 3.41)
B's velocity = 0.83 m/s
B's intended action = ...
```

A can simply see enough of the table/person/environment to act appropriately.

---

# 3. But what if A REALLY needs to know B?

Now we have an important distinction.

There are two possibilities.

### Case A — A can observe B

Then B's position might simply be part of A's observation:

$$
o_A =
[\text{A position},\text{B position},\text{box position}]
$$

No problem.

A **does know about B**.

---

### Case B — A cannot observe B

Then:

$$
o_A =
[\text{A position},\text{box}]
$$

A genuinely doesn't know where B is.

Now coordination is harder.

But it can still learn coordination through **interaction history**.

For example:

```text
t=0
A does X
B does Y
→ reward +10

t=1
A does X
B does Z
→ reward -10
```

Over many episodes, A can learn:

> "When I see this situation, doing X tends to work."

It doesn't necessarily learn:

> "B is currently at coordinate (4,7)."

It learns a policy based on what **it can actually observe**.

---

# 4. Here's where your intuition is 100% correct

If B's hidden state is **essential** for A's decision, then A has a problem.

Imagine:

```text
A sees:

       BOX
        |
        A
```

But B could secretly be either:

```text
Situation 1:

B → BOX ← A


Situation 2:

B
↓
BOX ← A
```

A sees exactly the same thing.

But the correct action is different depending on B.

A cannot reliably choose the correct action.

Why?

Because **the information needed to make the decision isn't in \(o_A\)**.

No algorithm can magically recover information that isn't observable.

This is one of the fundamental problems of **partial observability**.

---

# 5. So how does MAPPO help?

Here's the clever part.

During **training**, we can give the critic information that the actors don't have.

Suppose the actual situation is:

```text
                 GLOBAL WORLD

        B
        ↓
       BOX ← A
```

The actors receive:

```text
A:
o_A = what A can see

B:
o_B = what B can see
```

But the critic receives:

```text
s = entire world
```

So:

```text
               GLOBAL STATE
                    │
                    ▼
             Central Critic
                  V(s)
                    │
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
      Actor A                 Actor B
       o_A                     o_B
        ↓                       ↓
       a_A                     a_B
```

The critic knows:

> "Ah, B was actually behind the box."

A doesn't.

---

# 6. But wait — doesn't that STILL mean A can't coordinate?

**Yes, potentially.**

And this is extremely important:

> **CTDE does NOT magically solve partial observability.**

The centralized critic helps the **learning process**.

It does not give A hidden information at execution time.

If A genuinely needs B's hidden state to act correctly, you need something additional, such as:

* communication between agents
* better observations
* recurrent policies / memory
* belief states
* explicit communication actions

MAPPO itself isn't telepathy. 😭

---

# 7. Then why is the centralized critic useful?

Because training a partially observed multi-agent system is noisy as hell.

Consider A:

```text
A takes action LEFT
```

Then gets:

```text
reward = -10
```

Why?

Was A's action bad?

Or did B screw up?

Or was the environment already in a bad state?

A local critic only sees:

```text
o_A → V(o_A)
```

It doesn't have the whole picture.

Centralized critic sees:

```text
s
+
joint situation
→
V(s)
```

So it can produce a **better estimate of the team's situation**.

That gives A a better advantage estimate during training.

---

# 8. The key distinction

This is the thing I want you to lock in:

### Actor

Answers:

> **"Given what I can currently see, what should I do?"**

$$
a_i \sim \pi_i(a_i|o_i)
$$

### Critic

Answers:

> **"Given the whole situation, how good is this situation?"**

$$
V(s)
$$

So:

```text
EXECUTION:

A only gets → o_A → Actor A → a_A
B only gets → o_B → Actor B → a_B


TRAINING:

A's observation ─────→ Actor A
B's observation ─────→ Actor B
                           │
Global state ─────────→ Critic
                           │
                           ↓
                    better advantage
```

---

# 9. And now coordination makes more sense

Coordination doesn't necessarily mean:

> "A must know exactly what B is doing."

It can mean:

> "A and B have learned policies whose actions work well together."

For example, imagine traffic.

Two drivers don't need to know each other's neural-network internals.

They coordinate because they have:

* observations
* shared rules
* environment signals
* learned behavior
* possibly communication

Multi-agent RL is basically trying to learn this kind of **joint behavior**.

---

# 10. One final example

Imagine a game:

```text
A                B

🚗 →      🏁      ← 🚗
```

Reward:

```text
+100 if both reach the goal
-100 if they collide
```

A doesn't see B.

During training:

```text
A observation → Actor A → action A
B observation → Actor B → action B

                ↓
             Environment
                ↓
             reward
                ↓
       Global state → Critic
```

After millions of interactions, A may learn:

> "When I see this configuration, moving right tends to produce good team outcomes."

B independently learns its compatible behavior.

**That's coordination without direct knowledge of the other's internal state.**

But if the problem fundamentally requires hidden information from B, **we need communication or memory**. That's not a MAPPO failure; it's an information constraint.

---

if every agent gets the full global state, it's basically multi-agent PPO with centralized observations.

Classic MAPPO is specifically:

$$ \boxed{\text{Decentralized Actors} + \text{Centralized Critic}} $$

If actors also get the full state:

$$ \boxed{\text{Centralized Actors} + \text{Centralized Critic}} $$

So the PPO optimization machinery is still there, but you've removed the key decentralized-execution constraint that makes MAPPO interesting.

PettingZoo Multi-Agent Environment Setup

Yep. Now let's do the **actual 1.2 working**, one timestep at a time. No abstraction for the sake of abstraction.

# 1.2 — Actual Multi-Agent Environment Loop

We'll use a tiny 2-agent environment.

### Environment

```text
Position:
A = 2
B = 7
Target = 10
```

Each agent can choose:

```text
-1 = move left
 0 = stay
+1 = move right
```

Both agents are trying to reach the target.

---

## Step 0 — `reset()`

The environment starts:

$$
s_0=[2,7,10]
$$

Remember:

* `2` = A's position
* `7` = B's position
* `10` = target

We now generate each agent's observation.

Let's say we designed the environment so each agent sees **its own position + target**:

$$
o_A^0=[2,10]
$$

$$
o_B^0=[7,10]
$$

So the actors receive:

```text
A gets → [2, 10]
B gets → [7, 10]
```

They don't necessarily get the global state.

---

# Step 1 — Actors choose actions

Suppose the policies output:

$$
a_A^0=+1
$$

$$
a_B^0=+1
$$

Meaning both move right.

We send the **two actions together**:

```text
{
    A: +1,
    B: +1
}
```

This is the important difference from normal single-agent Gym.

---

# Step 2 — Environment executes them

The environment updates its state.

Our transition rule is:

$$
x_A^{t+1}=x_A^t+a_A^t
$$

$$
x_B^{t+1}=x_B^t+a_B^t
$$

Therefore:

$$
x_A^1=2+1=3
$$

$$
x_B^1=7+1=8
$$

Target remains 10.

So:

$$
\boxed{s_1=[3,8,10]}
$$

---

# Step 3 — Environment calculates reward

Let's make the reward:

$$
r_t=-\left(|x_A-target|+|x_B-target|\right)
$$

Before the action:

$$
|2-10|+|7-10|=8+3=11
$$

After the action:

$$
|3-10|+|8-10|=7+2=9
$$

So:

$$
\boxed{r_0=-9}
$$

Because we're using a shared team reward:

$$
r_A^0=r_B^0=-9
$$

The reward improved from \(-11\) to \(-9\), meaning the team moved closer.

---

# Step 4 — Generate new observations

The new state is:

$$
s_1=[3,8,10]
$$

Therefore:

$$
o_A^1=[3,10]
$$

$$
o_B^1=[8,10]
$$

Now we have completed **one RL timestep**.

Our collected transition is:

```text
t = 0

A observation = [2,10]
A action      = +1

B observation = [7,10]
B action      = +1

reward        = -9

next state    = [3,8,10]
```

---

# Step 5 — Do it again

At \(t=1\):

```text
A sees [3,10]
B sees [8,10]
```

Suppose:

$$
a_A=+1
$$

$$
a_B=+1
$$

Environment:

$$
x_A=3+1=4
$$

$$
x_B=8+1=9
$$

Therefore:

$$
s_2=[4,9,10]
$$

Reward:

$$
r_1=-(|4-10|+|9-10|)
$$

$$
=-7
$$

Again:

```text
A reward = -7
B reward = -7
```

And observations become:

$$
o_A^2=[4,10]
$$

$$
o_B^2=[9,10]
$$

---

# Step 6 — Eventually

Suppose:

```text
t=0 → reward -9
t=1 → reward -7
t=2 → reward -5
t=3 → reward -3
t=4 → reward ...
```

Eventually the agents reach the target.

The environment might then say:

```text
termination = True
```

and the episode ends.

---

# So what is PettingZoo doing?

**Basically this.**

We provide:

```text
actions:
{
    "agent_A": action_A,
    "agent_B": action_B
}
```

PettingZoo calls our environment's transition logic, and we return:

```text
observations
rewards
terminations
truncations
infos
```

The important mathematical loop is:

$$
\boxed{
s_t
\overset{a_A,a_B}{\longrightarrow}
s_{t+1}
}
$$

then:

$$
\boxed{
r_t=R(s_t,a_A,a_B,s_{t+1})
}
$$

then:

$$
\boxed{
s_{t+1}\rightarrow(o_A^{t+1},o_B^{t+1})
}
$$

And repeat.

---

## Where MAPPO eventually plugs in

For now:

```text
                ENVIRONMENT
                     │
              s_t → observations
                  ↙       ↘
               A           B
               ↓           ↓
              a_A         a_B
                  ↘       ↙
                 ENVIRONMENT
                     ↓
                  reward
```

Later, MAPPO adds:

```text
A observation → Actor A → a_A
B observation → Actor B → a_B

global state → Central Critic → V(s)
```


What Actually Happens in a Multi-Agent Environment

We already know:

\(s_t\) = entire world
\(o_A,o_B\) = what each agent is allowed to see
\(a_A,a_B\) = their actions

Now let's complete the actual RL loop.

1. One timestep

Suppose:

$$ s_t = [A=2,\ B=7,\ Target=10] $$

Observations:

$$ o_A=[2,10] $$ $$ o_B=[7,10] $$

Each actor produces an action:

$$ a_A=+1,\qquad a_B=+1 $$

The environment receives:

$$ \mathbf a_t=(a_A,a_B) $$

and changes:

$$ s_t \rightarrow s_{t+1} $$

Suppose:

$$ s_{t+1}=[3,8,10] $$
2. The environment calculates reward

This is not something MAPPO calculates.

The environment defines:

$$ r_t=R(s_t,\mathbf a_t,s_{t+1}) $$

For example:

$$ r_t=-\left(|A-Target|+|B-Target|\right) $$

After the movement:

$$ r_t=-(|3-10|+|8-10|) $$ $$ =-9 $$

If it's a cooperative task:

$$ r_A=r_B=-9 $$

So the environment has now produced:

new observations
reward
done/not done
3. Then the next timestep starts

The agents don't receive the old observations anymore.

They receive observations generated from the new state:

$$ s_{t+1}\rightarrow (o_A^{t+1},o_B^{t+1}) $$

So:

$$ o_A^{t+1}=[3,10] $$ $$ o_B^{t+1}=[8,10] $$

Then:

$$ (o_A^{t+1},o_B^{t+1}) \rightarrow (a_A^{t+1},a_B^{t+1}) $$

And the loop continues.

So the actual system is:

$$ \boxed{ s_t \rightarrow o_t^A,o_t^B \rightarrow a_t^A,a_t^B \rightarrow r_t,s_{t+1} \rightarrow o_{t+1}^A,o_{t+1}^B } $$

That's one multi-agent RL timestep.

4. What do we collect?

Now suppose we run for \(T=4\) timesteps.

We collect:

t	\(o_A\)	\(a_A\)	\(o_B\)	\(a_B\)	reward
0	[2,10]	+1	[7,10]	+1	-9
1	[3,10]	+1	[8,10]	+1	-7
2	[4,10]	+1	[9,10]	+1	-5
3	[5,10]	+1	[10,10]	0	...

This is our trajectory / rollout.

And this is what PPO needs to learn.

5. But MAPPO needs one extra thing

Remember the centralized critic?

It needs the global state.

So during training we also retain:

$$ s_0,s_1,s_2,s_3 $$

Our rollout therefore conceptually becomes:

t=0:
    s₀
    oA₀ → aA₀
    oB₀ → aB₀
    reward₀

t=1:
    s₁
    oA₁ → aA₁
    oB₁ → aB₁
    reward₁

t=2:
    s₂
    oA₂ → aA₂
    oB₂ → aB₂
    reward₂

Notice the distinction:

Actors need observations.

Critic gets global state.

6. Then PPO learning happens

This is where we return to your existing PPO knowledge.

After collecting the trajectory, we calculate the value:

$$ V(s_t) $$

and eventually the advantage:

$$ A_t $$

For example, with one-step TD:

$$ A_t=r_t+\gamma V(s_{t+1})-V(s_t) $$

In actual MAPPO we'll use GAE, which we'll cover in 1.4.

Then each actor gets a PPO update.

For Agent A:

$$ r_t^A(\theta) = \frac{ \pi_\theta^A(a_t^A|o_t^A) }{ \pi_{\text{old}}^A(a_t^A|o_t^A) } $$

For Agent B:

$$ r_t^B(\theta) = \frac{ \pi_\theta^B(a_t^B|o_t^B) }{ \pi_{\text{old}}^B(a_t^B|o_t^B) } $$

And the normal PPO clipped objective applies.

So the PPO mathematics isn't being replaced.

We're changing the information structure around it.

7. One thing that is easy to misunderstand

Suppose:

$$ r_t=-9 $$

Both agents receive \(-9\).

That does not mean:

"Agent A gets punished because A specifically made a bad move."

It means:

"The team was in a situation that produced this outcome."

That's why credit assignment becomes harder in multi-agent RL.

Later, the centralized critic helps provide a more informed estimate of how good the overall situation was.

This is one of the major reasons MAPPO exists.

8. Where PettingZoo fits

PettingZoo isn't the algorithm.

It's basically the standardized environment interface for the multi-agent world.

Conceptually:

                 PETTINGZOO ENV
                       │
        ┌──────────────┼──────────────┐
        ↓              ↓              ↓
   observations     rewards       termination
        │              │
        ↓              ↓
     Actors         Rollout
        │           Buffer
        ↓              │
     actions ──────────┘
              │
              ↓
         next environment state

It handles the interaction.

MAPPO handles learning.

9. One final distinction: environment vs agents

This is worth getting absolutely right.

Environment

Responsible for:

$$ s_{t+1}=f(s_t,a_A,a_B) $$

and:

$$ r_t=R(s_t,a_A,a_B,s_{t+1}) $$
Actors

Responsible for:

$$ a_A\sim\pi_A(a|o_A) $$ $$ a_B\sim\pi_B(a|o_B) $$
Critic

Responsible for estimating:

$$ V(s) $$
GAE/PPO

Responsible for using those collected quantities to update the policies.

The complete picture

At this point, the entire thing should look like:

                    ENVIRONMENT
                       │
                     state sₜ
                    /       \
                   ↓         ↓
                o_A         o_B
                 │           │
                 ↓           ↓
              Actor A     Actor B
                 │           │
                 ↓           ↓
                a_A         a_B
                   \         /
                    ↓       ↓
                    ENVIRONMENT
                         │
                    sₜ₊₁ + rₜ
                         │
                         ↓
                    next timestep


        During training only:

              sₜ
               ↓
        Central Critic
               ↓
             V(sₜ)
               ↓
             GAE
               ↓
          PPO updates
               ↓
         Actors improve

So 1.2 conceptually ends here.

We now understand:

Environment→multi-agent interaction→trajectory→centralized information→PPO training

What MAPPO actually stores

Now we're at the important connection.

Suppose we collect \(T=100\) timesteps.

For each timestep, we need things like:

$$ o_t^A,\quad o_t^B $$ $$ a_t^A,\quad a_t^B $$ $$ r_t $$

and, because we're using CTDE:

$$ s_t $$

So our rollout is conceptually:

$$ \boxed{ \{s_t,o_t^1,\ldots,o_t^N, a_t^1,\ldots,a_t^N,r_t,d_t\}_{t=0}^{T-1} } $$

That's the data that eventually gets fed into the MAPPO learning algorithm.

9. One subtle thing: the environment doesn't know PPO

This is important.

Our environment doesn't care whether we're using:

random actions
PPO
MAPPO
SAC
DQN
some completely different algorithm

It only understands:

$$ \text{actions} \rightarrow \text{world transition} \rightarrow \text{reward} $$

So:

            RL ALGORITHM
                 │
              actions
                 ↓
        ┌─────────────────┐
        │   PettingZoo    │
        │   Environment   │
        └─────────────────┘
                 │
        observations
        rewards
        termination
                 ↓
            RL ALGORITHM

From Environment to Critic Update

We've got:

$$ s_t \rightarrow o_t^A,o_t^B \rightarrow a_t^A,a_t^B \rightarrow r_t,s_{t+1} $$

Now what happens after collecting those transitions?

1. We collect a rollout

Say we run the environment for 3 steps:

\(t\)	\(o_A\)	\(a_A\)	\(o_B\)	\(a_B\)	\(r_t\)	\(s_t\)
0	\(o_A^0\)	\(a_A^0\)	\(o_B^0\)	\(a_B^0\)	2	\(s_0\)
1	\(o_A^1\)	\(a_A^1\)	\(o_B^1\)	\(a_B^1\)	4	\(s_1\)
2	\(o_A^2\)	\(a_A^2\)	\(o_B^2\)	\(a_B^2\)	10	\(s_2\)

Now the central critic sees the global states:

$$ s_0,s_1,s_2 $$

and predicts:

$$ V_\phi(s_0),V_\phi(s_1),V_\phi(s_2) $$

Suppose it predicts:

$$ V(s_0)=6 $$ $$ V(s_1)=7 $$ $$ V(s_2)=8 $$

These are predictions of:

"How much future discounted reward do I expect from this state?"

2. The critic needs a target

The critic currently thinks:

$$ V(s_0)=6 $$

But we have actually experienced:

$$ r_0=2,\quad r_1=4,\quad r_2=10 $$

So we need to calculate what the value should have been.

For a simple one-step target:

$$ y_t=r_t+\gamma V(s_{t+1}) $$

Take:

$$ \gamma=0.9 $$

For \(t=0\):

$$ y_0=2+0.9(7) $$ $$ y_0=8.3 $$

So the critic predicted:

$$ 6 $$

but our target says:

$$ 8.3 $$

The critic needs to move its prediction toward 8.3.

3. This is where the TD error appears

Define:

$$ \delta_t= r_t+\gamma V(s_{t+1})-V(s_t) $$

For \(t=0\):

$$ \delta_0 = 2+0.9(7)-6 $$ $$ \boxed{\delta_0=2.3} $$

Positive TD error means:

"The outcome was better than the critic expected."

If it were negative:

"The outcome was worse than expected."

This is the basic learning signal for the critic.

4. But MAPPO uses GAE

Instead of simply using \(\delta_t\), we'll combine multiple future TD errors.

GAE:

$$ \boxed{ A_t= \delta_t+ \gamma\lambda\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} +\cdots } $$

So the advantage answers:

"Was the action taken here better or worse than what the critic expected?"

This is crucial because the actor needs an advantage, not merely a raw reward.

We'll do the full numerical GAE calculation in Lesson 1.4.

5. Where the critic update happens

The critic has:

$$ V_\phi(s_t) $$

and a target, typically:

$$ \hat V_t=A_t+V_\phi(s_t) $$

Then its loss is essentially:

$$ \boxed{ L_V= \frac12 \left( V_\phi(s_t)-\hat V_t \right)^2 } $$

So if:

critic predicted = 6
target            = 8.3

the loss pushes the critic toward 8.3.

Gradient descent updates its parameters:

$$ \phi \leftarrow \phi-\alpha_v\nabla_\phi L_V $$

That's the critic update.

6. Now the actors get updated

Suppose Agent A previously chose:

$$ a_A^t $$

Its old policy probability was:

$$ \pi_{\text{old}}(a_A^t|o_A^t)=0.2 $$

After changing the network, suppose:

$$ \pi_\theta(a_A^t|o_A^t)=0.3 $$

PPO computes:

$$ r_t^A= \frac{0.3}{0.2} = 1.5 $$

Then uses the advantage:

$$ A_t $$

inside the clipped PPO objective:

$$ L_A= \min \left( r_t^A A_t, \operatorname{clip}(r_t^A,1-\epsilon,1+\epsilon)A_t \right) $$

Same for Agent B.

7. Why the centralized critic matters

Here's the entire reason we're doing this.

Suppose:

$$ r_t=+10 $$

Both agents contributed actions.

Agent A sees only:

$$ o_A $$

Agent B sees only:

$$ o_B $$

But the critic sees:

$$ s_t $$

which contains the whole situation.

So the critic learns:

$$ V(s_t) $$

and produces a team-level baseline.

Then both agents can use that advantage:

$$ A_t $$

to update their policies.

Conceptually:

             GLOBAL STATE
                  ↓
          Central Critic
                  ↓
                V(s)
                  ↓
          ┌───────┴───────┐
          ↓               ↓
       Agent A          Agent B
      advantage A      advantage A
          ↓               ↓
       PPO update       PPO update

For shared rewards, the same team advantage can be used for both actors.

8. So the COMPLETE MAPPO training iteration is

This is the part we should have established before talking about code:

Rollout
$$ s_t \rightarrow o_t^A,o_t^B \rightarrow a_t^A,a_t^B \rightarrow r_t,s_{t+1} $$

Repeat for \(T\) steps.

Critic
$$ s_t\rightarrow V_\phi(s_t) $$
GAE
$$ (r_t,V_t,V_{t+1}) \rightarrow \delta_t \rightarrow A_t $$
Actor
$$ (o_t^i,a_t^i,A_t) \rightarrow \text{PPO ratio} \rightarrow \text{clipped policy loss} $$
Updates
$$ \phi\leftarrow\phi-\alpha_v\nabla L_V $$ $$ \theta_i\leftarrow\theta_i-\alpha_\pi\nabla L_{\pi_i} $$

Then throw away/refresh the rollout and collect another one.

9. The whole thing in one picture
                    ENVIRONMENT
                         │
                       sₜ
                    ↙       ↘
                 o_A         o_B
                  ↓           ↓
               Actor A     Actor B
                  ↓           ↓
                 a_A         a_B
                    ↘       ↙
                      ENV
                       │
                 rₜ , sₜ₊₁
                       │
                       ↓
                  COLLECT DATA
                       │
          ┌────────────┴────────────┐
          ↓                         ↓
     Global states              Local data
          ↓                         ↓
   Central Critic              Actor ratios
          ↓                         ↓
       V(sₜ)                    PPO loss
          ↓                         ↓
         GAE                   Actor update
          │
          └────────────┐
                       ↓
                 Critic update
                       │
                       ↓
                 NEXT ROLLOUT

Decentralized Actors + Centralized Critic

Now we're at the actual MAPPO architecture. The key question is:

What exactly are the networks, what do they receive, and what gets updated?

1. The architecture

For 2 agents, we have:

$$ \pi_{\theta_A}(a_A|o_A) $$ $$ \pi_{\theta_B}(a_B|o_B) $$

and one centralized critic:

$$ V_\phi(s) $$

So:

A observation ──→ Actor A ──→ action A
                       │
B observation ──→ Actor B ──→ action B

Global state ────────→ Critic ──→ V(s)

The actors never need the global state.

The critic doesn't choose actions.

2. What does an actor actually output?

Suppose each agent has 5 possible actions:

$$ A=\{0,1,2,3,4\} $$

Agent A sees:

$$ o_A=[2,10] $$

Its neural network produces 5 logits:

$$ z_A=[1.2,-0.4,0.7,2.1,0.1] $$

Softmax converts those into probabilities:

$$ \pi_A(a|o_A) $$

For example:

up      0.08
down    0.02
left    0.05
right   0.80
stay    0.05

We sample:

$$ a_A=\text{right} $$

That's literally the actor.

Same process for B.

3. What does the critic output?

The critic gets the global state.

Suppose:

$$ s=[A_x,A_y,B_x,B_y,T_x,T_y] $$

Example:

$$ s=[0,1,3,4,2,2] $$

The critic is simply a neural network:

$$ V_\phi(s) $$

and produces one scalar:

$$ V_\phi(s)=7.3 $$

That's it.

Actor:

observation → probability distribution → action

Critic:

global state → one number
4. Why separate actors?

Because each agent has its own observation and potentially its own policy:

$$ \pi_A(a_A|o_A) $$ $$ \pi_B(a_B|o_B) $$

They can therefore learn different behaviors.

For example:

A = defender
B = attacker

could have completely different policies.

5. But do we REALLY need separate networks?

No.

This is an important engineering choice.

Option 1 — Independent actors
Actor A → parameters θA
Actor B → parameters θB

They learn independently.

Option 2 — Parameter sharing
             Shared Actor
            /            \
         o_A              o_B
          ↓                ↓
        action A         action B

Same parameters:

$$ \theta_A=\theta_B=\theta $$

This is often useful when agents are homogeneous.

For our first MAPPO implementation, we'll use parameter sharing.

Why?

Because otherwise we're doubling the actor parameters without gaining anything for our symmetric toy agents.

6. The really important part: what happens during one update?

Suppose our rollout contains:

$$ (o_t^A,o_t^B,a_t^A,a_t^B,s_t,A_t) $$

For Agent A we calculate:

$$ \pi_\theta(a_t^A|o_t^A) $$

For Agent B:

$$ \pi_\theta(a_t^B|o_t^B) $$

Then:

$$ r_t^A= \frac{\pi_\theta(a_t^A|o_t^A)} {\pi_{\text{old}}(a_t^A|o_t^A)} $$

and:

$$ r_t^B= \frac{\pi_\theta(a_t^B|o_t^B)} {\pi_{\text{old}}(a_t^B|o_t^B)} $$

Notice:

Each actor's ratio uses its own observation and its own action.

7. But they can share the same advantage

For cooperative MAPPO with a shared team reward, we can calculate:

$$ A_t $$

from the centralized critic.

Then both agents use it:

$$ L_A \propto r_t^A A_t $$ $$ L_B \propto r_t^B A_t $$

This tells both policies:

"Whatever you did at this timestep contributed to a team outcome that was better/worse than expected."

8. Critic update

Separately, the critic gets:

$$ s_t $$

and predicts:

$$ V_\phi(s_t) $$

We have a target:

$$ \hat V_t $$

Then:

$$ L_V= \frac12 (V_\phi(s_t)-\hat V_t)^2 $$

Gradient descent:

$$ \phi\leftarrow \phi-\alpha_v\nabla_\phi L_V $$

So there are two learning systems:

Actors:
observation → action probabilities
              ↓
          PPO loss
              ↓
        actor parameters


Critic:
global state → V(s)
               ↓
          value loss
               ↓
        critic parameters
9. Gradient flow — VERY important

The critic's parameters:

$$ \phi $$

are updated from the value loss.

The actor's parameters:

$$ \theta $$

are updated from the policy loss.

They are not one giant network where everything gets updated by everything.

Conceptually:

             rollout
                │
        ┌───────┴────────┐
        ↓                ↓
      Actor             Critic
   o_i → π(a|o)        s → V(s)
        ↓                ↓
   policy loss         value loss
        ↓                ↓
       θ                φ

The critic influences the actor through the advantage calculation, not by directly sending gradients through the actor.

10. Why this is called CTDE

During training:

$$ \boxed{ \text{Actor: local information} } $$ $$ \boxed{ \text{Critic: global information} } $$

During execution:

A:
o_A → Actor → a_A

B:
o_B → Actor → a_B

The critic isn't required.

Therefore:

Centralized Training+Decentralized Execution
	​


Option 1 — Two completely separate actors

You have:

$$ \pi_{\theta_A}(a_A|o_A) $$ $$ \pi_{\theta_B}(a_B|o_B) $$

Two networks, two parameter sets:

o_A → Actor A (θA) → action A
o_B → Actor B (θB) → action B

They are trained separately, although they can use the same team advantage \(A_t\).

There is nothing to merge between their outputs.

The environment simply receives both actions:

$$ (a_A,a_B) $$

and acts on the joint action.

Option 2 — One shared actor

Here there is literally one network:

$$ \pi_\theta(a|o) $$

You feed A's observation:

$$ o_A \rightarrow \pi_\theta \rightarrow P_A $$

and B's observation:

$$ o_B \rightarrow \pi_\theta \rightarrow P_B $$

Because \(o_A\neq o_B\), the outputs can be completely different despite identical parameters.

Example:

                  SAME NETWORK θ
                 /              \
             o_A                  o_B
              ↓                    ↓
        [0.1,0.1,0.7,0.1]    [0.6,0.1,0.1,0.2]
              ↓                    ↓
          action A              action B

So there aren't actually "two personalities" stored in the network.

The different observations produce different outputs.

"But how do we merge the outputs?"

We don't.

This is the key point.

The outputs remain separate:

$$ \pi_\theta(o_A)=P_A $$ $$ \pi_\theta(o_B)=P_B $$

We sample:

$$ a_A\sim P_A $$ $$ a_B\sim P_B $$

Then send:

$$ \boxed{(a_A,a_B)} $$

to the environment.

The environment handles the interaction between them.

And during training?

This is where the shared network becomes interesting.

Suppose:

$$ A_t=+2 $$

meaning the team's outcome was better than expected.

Both agents' experiences contribute to updating the same \(\theta\):

$$ L_{\text{actor}} = L_A+L_B $$

Conceptually:

A experience ──→ PPO loss A ──┐
                              ├──→ shared θ update
B experience ──→ PPO loss B ──┘

So the network learns from both agents.

With separate networks:

A experience → loss A → θA

B experience → loss B → θB

No parameter sharing.

So your two choices are:
	Shared actor	Separate actors
Networks	1	2
Parameters	Same	Different
Input	each agent's own \(o_i\)	each agent's own \(o_i\)
Output	separate distribution per agent	separate distribution per agent
Merge outputs?	No	No
Training	both experiences update same \(\theta\)	each updates its own \(\theta_i\)
Good for	homogeneous agents	heterogeneous agents

GAE + MAPPO Training Loop

This is the final piece of classical MAPPO. We'll do it with an actual trajectory so the math connects to what the networks are doing.

1. Start with a rollout

Two agents, shared reward.

Suppose we collected:

\(t\)	reward \(r_t\)	critic \(V(s_t)\)
0	2	6.0
1	4	7.0
2	10	8.0
3	—	9.0

The last \(V(s_3)=9\) is important: it's the critic's estimate of what happens after timestep 2.

Use:

$$ \gamma=0.9,\qquad \lambda=0.95 $$
2. First calculate TD error

The TD error is:

$$ \boxed{ \delta_t=r_t+\gamma V(s_{t+1})-V(s_t) } $$

For \(t=0\):

$$ \delta_0=2+0.9(7)-6 $$ $$ \boxed{\delta_0=2.3} $$

For \(t=1\):

$$ \delta_1=4+0.9(8)-7 $$ $$ \boxed{\delta_1=4.2} $$

For \(t=2\):

$$ \delta_2=10+0.9(9)-8 $$ $$ \boxed{\delta_2=10.1} $$

So:

$$ [\delta_0,\delta_1,\delta_2] = [2.3,4.2,10.1] $$
3. Now GAE

This is the part people tend to make unnecessarily mysterious.

GAE says:

Don't judge an action using only its immediate TD error. Also consider subsequent TD errors, but discount their influence.

The formula:

$$ \boxed{ A_t= \delta_t+ \gamma\lambda\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} +\cdots } $$

Since:

$$ \gamma\lambda=0.9\times0.95=0.855 $$
At \(t=2\)

Nothing comes after it:

$$ A_2=\delta_2=10.1 $$
At \(t=1\)
$$ A_1 = 4.2+0.855(10.1) $$ $$ \boxed{A_1=12.8355} $$
At \(t=0\)
$$ A_0 = 2.3+0.855(4.2)+0.855^2(10.1) $$ $$ \boxed{A_0\approx13.37} $$

So our advantages are approximately:

$$ \boxed{ A=[13.37,\ 12.84,\ 10.1] } $$

All positive.

Meaning:

The actions taken in these states turned out better than what the critic expected.

4. What does the actor do with this?

Suppose at \(t=0\), Agent A chose:

$$ a_A^0 $$

The old policy gave that action probability:

$$ \pi_{\text{old}}(a_A^0|o_A^0)=0.20 $$

After the network update we're evaluating:

$$ \pi_\theta(a_A^0|o_A^0)=0.24 $$

PPO ratio:

$$ r_0= \frac{0.24}{0.20} =1.2 $$

Our advantage is:

$$ A_0=13.37 $$

Therefore the unclipped objective is:

$$ 1.2(13.37)=16.044 $$

With:

$$ \epsilon=0.2 $$

the ratio is still inside:

$$ [0.8,1.2] $$

so PPO allows that increase.

The actor therefore gets pushed toward making that action more likely.

5. What about the critic?

The critic predicted:

$$ V(s_0)=6 $$

We need a target.

GAE gives us the advantage, so a value target can be constructed as:

$$ \boxed{ \hat V_t=A_t+V(s_t) } $$

Therefore:

$$ \hat V_0=13.37+6 $$ $$ \boxed{\hat V_0=19.37} $$

The critic predicted 6, but our rollout indicates a much higher value.

So the critic loss is:

$$ L_V= \frac12 (V(s_t)-\hat V_t)^2 $$

For \(t=0\):

$$ L_V= \frac12(6-19.37)^2 $$

The gradient pushes the critic's prediction upward.

6. Why doesn't the actor just use reward directly?

Because reward:

$$ r_t $$

only tells you what happened immediately.

Advantage asks:

$$ \boxed{ \text{"Was this action better or worse than expected?"} } $$

That's a much better signal for policy learning.

Imagine:

action → small immediate reward
        → leads to huge future reward

GAE can propagate information about that future outcome backward.

That's one of its main jobs.

7. Now the actual MAPPO training iteration

This is the full thing.

Phase 1 — Rollout

Run the current policies:

$$ o_t^i\rightarrow\pi_\theta\rightarrow a_t^i $$

Environment:

$$ (a_t^1,\ldots,a_t^N) \rightarrow r_t,s_{t+1} $$

Store:

$$ (s_t,o_t^i,a_t^i,r_t) $$

for every timestep.

Phase 2 — Critic evaluation

Feed global states:

$$ s_t\rightarrow V_\phi(s_t) $$

giving:

$$ V_0,V_1,\ldots,V_T $$
Phase 3 — GAE

Calculate:

$$ \delta_t=r_t+\gamma V_{t+1}-V_t $$

then:

$$ A_t=\delta_t+\gamma\lambda A_{t+1} $$

The recursive form is what we'll actually implement.

Phase 4 — Actor update

For each agent:

$$ r_t^i= \frac{ \pi_\theta(a_t^i|o_t^i) }{ \pi_{\text{old}}(a_t^i|o_t^i) } $$

Then:

$$ L_{\pi_i} = -\min \left( r_t^iA_t, \operatorname{clip}(r_t^i,1-\epsilon,1+\epsilon)A_t \right) $$

If using a shared actor:

$$ \boxed{ L_\pi=\frac{1}{N}\sum_iL_{\pi_i} } $$

Then:

$$ \theta\leftarrow\theta-\alpha_\pi\nabla_\theta L_\pi $$
Phase 5 — Critic update
$$ L_V= \frac12(V_\phi(s_t)-\hat V_t)^2 $$

Then:

$$ \phi\leftarrow \phi-\alpha_V\nabla_\phi L_V $$
8. Then repeat

The old policy becomes the current policy:

$$ \pi_{\text{old}}\leftarrow\pi_\theta $$

Then collect another rollout.

So:

ROLLOUT
   ↓
critic values
   ↓
GAE
   ↓
actor update
   ↓
critic update
   ↓
new policy
   ↓
ROLLOUT AGAIN
   ↓
...

That's the classical MAPPO training loop.

9. Where the "variance" problem comes in

Here's the important intuition.

Suppose Agent A and B take actions simultaneously.

You observe:

$$ r_t=+10 $$

But who caused that +10?

Maybe:

A made an excellent action
B made a terrible action

or:

A made a terrible action
B made an excellent action

Yet both experienced the same team reward.

This creates noisy credit assignment.

The centralized critic helps because it sees:

$$ s_t $$

rather than just one agent's partial view.

GAE further reduces the variance of the advantage estimate by averaging information across multiple future TD errors through \(\lambda\).

PPO clipping prevents a single noisy advantage estimate from causing an enormous policy jump.

So three pieces work together:

$$ \boxed{ \text{Centralized critic} \rightarrow \text{better baseline} } $$ $$ \boxed{ \text{GAE} \rightarrow \text{lower-variance advantage estimate} } $$ $$ \boxed{ \text{PPO clipping} \rightarrow \text{bounded policy update} } $$
10. What "convergence" actually means

Don't think:

"The loss reaches zero."

That's not generally what we want.

For RL, we're looking for the behavior/performance to stabilize.

For example:

Episode return

-150
-120
 -80
 -40
 -15
 -10
 -11
 -10
 -10

The policy has basically stopped improving.

We'd also monitor things like:

mean episode return
episode length
policy entropy
approximate KL
value loss
advantage statistics
clip fraction

If return improves but entropy collapses instantly, that's suspicious.

If value loss explodes, critic training is probably unstable.

If KL becomes huge, the policy may be moving too aggressively.

The complete classical MAPPO picture

You should now be able to trace one piece of data all the way through:

$$ \boxed{ s_t \rightarrow o_t^A,o_t^B \rightarrow a_t^A,a_t^B \rightarrow r_t,s_{t+1} } $$

then:

$$ s_t\rightarrow V(s_t) $$

then:

$$ r_t,V_t,V_{t+1} \rightarrow\delta_t \rightarrow A_t $$

then:

$$ (o_t^i,a_t^i,A_t) \rightarrow \text{PPO loss} \rightarrow \theta $$

and:

$$ (s_t,\hat V_t) \rightarrow \text{value loss} \rightarrow \phi $$

That's classical MAPPO.